## Oleg G

# 1a

Under risk-neutral probabilities, $dr_t = \alpha(r,t)\,dt + \beta(r,t)\,dW_t$.
If $C_t = C(r_t,t)$, then by Ito's:

$$dC = \Big(\frac{\partial C}{\partial t} + \alpha\frac{\partial C}{\partial r} + \tfrac{1}{2}\beta^2\frac{\partial^2 C}{\partial r^2}\Big)dt + \beta\frac{\partial C}{\partial r}\,dW_t.$$

Since $C$ is the price of a traded derivative, its risk-neutral drift must equal $rC$. Setting the $dt$ coefficient equal to $rC$:

$$\;\frac{\partial C}{\partial t} + \alpha(r,t)\frac{\partial C}{\partial r} + \tfrac{1}{2}\beta(r,t)^2\frac{\partial^2 C}{\partial r^2} = rC\;$$

with terminal condition $C(r,T) = F(r)$.

For a $T$-maturity zero-coupon bond, $F(r_T) \equiv 1$, so $C(r,T) = 1$.

Under the Vasicek model $\alpha = \kappa(\theta - r)$, $\beta = \sigma$ (constant), the PDE becomes:

$$\frac{\partial C}{\partial t} + \kappa(\theta - r)\frac{\partial C}{\partial r} + \tfrac{1}{2}\sigma^2\frac{\partial^2 C}{\partial r^2} = rC.$$

In [29]:
import numpy as np

In [30]:
class Vasicek:

    def __init__(self,kappa,theta,sigma):
        self.kappa=kappa
        self.theta=theta
        self.sigma=sigma

In [31]:
hw41dynamics = Vasicek(kappa=3,theta=0.05,sigma=0.03)

In [32]:
class Bond:

    def __init__(self, T):
        self.T=T


In [33]:
hw41contract = Bond(T=5)

In [34]:
class FDexplicitEngine:

    def __init__(self, rMax, rMin, deltar, deltat, useUpwind):
        self.rMax=rMax
        self.rMin=rMin
        self.deltar=deltar
        self.deltat=deltat
        self.useUpwind=useUpwind

    def price_bond_vasicek(self,contract,dynamics):

        T = contract.T
        N=round(T/self.deltat)
        if abs(N-T/self.deltat) > 1e-12:
            raise ValueError("Bad delta t")

        r=np.arange(self.rMax,self.rMin-self.deltar/2,-self.deltar)   # index 0 = HIGH r
        bondprice=np.ones(np.size(r))

        kappa = dynamics.kappa
        theta = dynamics.theta
        sigma = dynamics.sigma
        dr = self.deltar
        dt = self.deltat

        alpha = kappa * (theta - r)   # drift array (note: in index convention, alpha[0] is for highest r)
        diff = sigma**2 * dt / (dr**2)   # diffusion coefficient factor

        # qu multiplies bondprice[:-2]
        # qd multiplies bondprice[2:]
        # qm multiplies bondprice[1:-1]

        if self.useUpwind:
            # Upwind: forward diff in r when drift>=0, backward diff when drift<0.
            qu = np.where(alpha >= 0,
                          alpha*dt/dr + 0.5*diff,
                          0.5*diff)
            qd = np.where(alpha >= 0,
                          0.5*diff,
                          -alpha*dt/dr + 0.5*diff)
            qm = 1 - qu - qd
        else:
            # Central difference in r
            qu = alpha*dt/(2*dr) + 0.5*diff
            qd = -alpha*dt/(2*dr) + 0.5*diff
            qm = 1 - diff * np.ones_like(r)

        for t in np.arange(N-1,-1,-1)*self.deltat:
            bondprice[1:-1]=1/(1+r[1:-1]*self.deltat)*(qd[1:-1]*bondprice[2:]+qm[1:-1]*bondprice[1:-1]+qu[1:-1]*bondprice[:-2])
            bondprice[0]=2*bondprice[1]-bondprice[2]
            bondprice[-1]=2*bondprice[-2]-bondprice[-3]

        return (r, bondprice)

In [35]:
hw41FD = FDexplicitEngine(rMax=0.35,rMin=-0.25,deltar=0.01,deltat=0.01,useUpwind=False)

In [36]:
(r, bondprice) = hw41FD.price_bond_vasicek(hw41contract,hw41dynamics)

In [37]:
np.set_printoptions(precision=4,suppress=True)
displayrows=(r<0.15+hw41FD.deltar/2) & (r>0.0-hw41FD.deltar/2)

In [38]:
print(np.stack((r, bondprice),axis=1)[displayrows])

[[ 1.5000e-01 -1.4273e+09]
 [ 1.4000e-01  1.6361e+08]
 [ 1.3000e-01  2.2294e+07]
 [ 1.2000e-01 -1.3724e+06]
 [ 1.1000e-01 -1.3361e+05]
 [ 1.0000e-01  3.2966e+03]
 [ 9.0000e-02  1.3021e+02]
 [ 8.0000e-02  7.7128e-01]
 [ 7.0000e-02  7.7385e-01]
 [ 6.0000e-02  7.7643e-01]
 [ 5.0000e-02  7.7902e-01]
 [ 4.0000e-02  7.8162e-01]
 [ 3.0000e-02  7.8423e-01]
 [ 2.0000e-02  7.8685e-01]
 [ 1.0000e-02  1.4165e+03]
 [-3.3307e-16  5.1498e+04]]


# 1d

Assume $f$ is smooth near $x$.

**Forward difference.** Taylor's theorem gives $f(x+h) = f(x) + f'(x)h + \tfrac{1}{2}f''(\xi_1)h^2$ for some $\xi_1$ between $x$ and $x+h$. Therefore

$$\frac{f(x+h)-f(x)}{h} - f'(x) = \tfrac{1}{2}f''(\xi_1)\,h = O(h).$$

**Central difference.** Expanding to third order:

$$f(x+h) = f(x) + f'(x)h + \tfrac{1}{2}f''(x)h^2 + \tfrac{1}{6}f'''(\xi_1)h^3,$$
$$f(x-h) = f(x) - f'(x)h + \tfrac{1}{2}f''(x)h^2 - \tfrac{1}{6}f'''(\xi_2)h^3.$$

Subtracting and dividing by $2h$:

$$\frac{f(x+h)-f(x-h)}{2h} - f'(x) = \frac{f'''(\xi_1)+f'''(\xi_2)}{12}\,h^2 = O(h^2).$$

So the central difference is one order more accurate than the one-sided (forward/upwind) difference.

# 1e

In [39]:
# Central difference
hw41FD_central = FDexplicitEngine(rMax=0.35, rMin=-0.25, deltar=0.01, deltat=0.01, useUpwind=False)
r_c, bp_c = hw41FD_central.price_bond_vasicek(hw41contract, hw41dynamics)

# Upwind
hw41FD_upwind = FDexplicitEngine(rMax=0.35, rMin=-0.25, deltar=0.01, deltat=0.01, useUpwind=True)
r_u, bp_u = hw41FD_upwind.price_bond_vasicek(hw41contract, hw41dynamics)

# Compare at r_0 = 0.10
idx = np.argmin(np.abs(r_c - 0.10))
print(f"Central-difference price at r_0 = 0.10: {bp_c[idx]:.6f}")
print(f"Upwind price at r_0 = 0.10:            {bp_u[idx]:.6f}")
print(f"Exact price:                            0.7661")

Central-difference price at r_0 = 0.10: 3296.592924
Upwind price at r_0 = 0.10:            0.766225
Exact price:                            0.7661


# 1f

Ignoring stability issues and considering only consistency (truncation error), the upwind explicit scheme, which uses one-sided spatial differences, discretizes the PDE with **less** accuracy than the standard explicit scheme, which uses central spatial differences.

However, to actually guarantee convergence, the grid spacing must satisfy certain stability constraints, to prevent errors from propagating explosively. In a PDE exhibiting strong drift, we have seen that these constraints may allow the upwind scheme **more** freedom in choosing grid spacing, compared to the central scheme.

# 1g

Use the central-difference (more accurate) result. Continuously-compounded yield of a $T$-maturity zero paying 1 at $T$:

$$y(0,T) = \frac{\log(1/P_0)}{T-0} = -\frac{\log P_0}{T}.$$

In [40]:
# Use the central-difference result
T_bond = hw41contract.T

for r0 in [0.12, 0.02]:
    idx = np.argmin(np.abs(r_c - r0))
    P0 = bp_c[idx]
    ytm = -np.log(P0) / T_bond
    print(f"r_0 = {r0:.2f}:  P_0 = {P0:.6f},  yield-to-maturity = {ytm:.6f}")

r_0 = 0.12:  P_0 = -1372394.781066,  yield-to-maturity = nan
r_0 = 0.02:  P_0 = 0.786850,  yield-to-maturity = 0.047943


/var/folders/x3/2h269p1576s293vgwkccn0zm0000gn/T/ipykernel_59042/1298029409.py:7: RuntimeWarning: invalid value encountered in log
  ytm = -np.log(P0) / T_bond


The Vasicek model is mean-reverting to $\theta = 0.05$. If $r_0 = 0.12 > \theta$, future short rates tend to drift *down* toward $0.05$, so the average of the instantaneous rates over $[0,T]$ is below $0.12$, and therefore the yield (essentially a time-average of expected future rates) is less than $0.12$. Symmetrically, if $r_0 = 0.02 < \theta$, future rates drift *up* toward $0.05$, so the average is above $0.02$, and the yield exceeds $0.02$.

## Problem 2

# 2a

Under risk-neutral probabilities, $X_t$ (a futures price) has zero drift:
$$dX_t = \sigma X_t^{1+\alpha}\,dW_t.$$

By Itô and the usual no-arbitrage argument, any European option price $C(X,t)$ satisfies
$$\frac{\partial C}{\partial t} \;+\; \tfrac{1}{2}\sigma^2 X^{2(1+\alpha)}\frac{\partial^2 C}{\partial X^2} \;=\; r\,C,$$

with terminal condition $C(X,T) = (K - X)^+$. (Note: the drift term $\mu X\,\partial C/\partial X$ is absent because the risk-neutral drift of the futures is zero; the option price itself still discounts at rate $r$.)

In [41]:
import numpy as np
from scipy.sparse import diags
from scipy.sparse.linalg import spsolve

In [42]:
class CEV:

    def __init__(self,volcoeff,alpha,rGrow,r,X0):
        self.volcoeff = volcoeff
        self.alpha = alpha
        self.rGrow = rGrow
        self.r = r
        self.X0 = X0


In [43]:
hw42dynamics = CEV(volcoeff=3, alpha=-0.5, rGrow=0, r=0.05, X0=100)

In [44]:
class Put:

    def __init__(self,T,K):
        self.T = T;
        self.K = K;

In [45]:
hw42contract = Put(T=0.25, K=100)

In [46]:
class FD_CrankNicolson_Engine:

    def __init__(self,XMax,XMin,deltaX,deltat):
        self.XMax=XMax
        self.XMin=XMin
        self.deltaX=deltaX
        self.deltat=deltat

    def TicksAndMatricesCEV(self,T,dynamics):

        alpha, r, rGrow, volcoeff = dynamics.alpha, dynamics.r, dynamics.rGrow, dynamics.volcoeff

        N=round(T/self.deltat)
        if abs(N-T/self.deltat)>1e-12:
            raise ValueError('Bad time step')
        numX=round((self.XMax-self.XMin)/self.deltaX)+1
        if abs(numX-(self.XMax-self.XMin)/self.deltaX-1)>1e-12:
            raise ValueError('Bad time step')
        X=np.linspace(self.XMax,self.XMin,numX)    #The FIRST indices in this array are for HIGH levels of X
        tTicks = np.arange(N-1,-1,-1)*self.deltat

        ratio1 = self.deltat/self.deltaX
        ratio2 = self.deltat/self.deltaX**2

        # PDE (written as dC/dtau = f C_XX + g C_X + h C, with tau = T-t):
        #   f = (1/2) * volcoeff^2 * X^(2(1+alpha))
        #   g = rGrow * X            (= 0 for a futures/forward; nonzero for BS stock)
        #   h = -r
        f = 0.5 * volcoeff**2 * X**(2.0*(1.0+alpha))
        g = rGrow * X
        h = -r  # scalar ok; broadcasts

        F = 0.5*ratio2*f + 0.25*ratio1*g
        G =     ratio2*f - 0.50*self.deltat*h
        H = 0.5*ratio2*f - 0.25*ratio1*g

        RHSmatrix = diags([H[:-1], 1-G, F[1:]], [1,0,-1], shape=(numX,numX), format="csr")
        LHSmatrix = diags([-H[:-1], 1+G, -F[1:]], [1,0,-1], shape=(numX,numX), format="csr")

        return(X, tTicks, LHSmatrix, RHSmatrix, H[-1], F[0])


    def price_put_CEV(self,contract,dynamics):

        X, tTicks, LHSmatrix, RHSmatrix, bottomH, topF = self.TicksAndMatricesCEV(contract.T,dynamics)

        putprice=np.maximum(contract.K-X,0)
        X_lowboundary=self.XMin-self.deltaX

        for t in tTicks:

            rhs = RHSmatrix @ putprice
            rhs[-1]=rhs[-1]+2*bottomH*(contract.K-X_lowboundary)

            putprice = spsolve(LHSmatrix, rhs)

            # American early exercise
            putprice = np.maximum(putprice, contract.K-X)

        return(X, putprice)

In [47]:
hw42FD = FD_CrankNicolson_Engine(XMax=200,XMin=50,deltaX=0.1,deltat=0.0005)

In [48]:
(X0_all, putprice) = hw42FD.price_put_CEV(hw42contract,hw42dynamics)

# 2c

Finite-difference estimates using neighboring grid values (central difference with $\Delta X = 0.1$).

In [49]:
# Find the index of X0=100 in X0_all
i100 = np.argmin(np.abs(X0_all - 100.0))
# Neighbors: X0_all decreases as index increases, so i100-1 is higher X, i100+1 is lower X
C_up   = putprice[i100 - 1]   # at X = 100 + dX
C_mid  = putprice[i100]        # at X = 100
C_down = putprice[i100 + 1]    # at X = 100 - dX

dX = hw42FD.deltaX

delta_CEV = (C_up - C_down) / (2 * dX)
gamma_CEV = (C_up - 2*C_mid + C_down) / (dX ** 2)

print(f"American put (CEV dynamics) at X0=100:")
print(f"  price = {C_mid:.6f}")
print(f"  delta = {delta_CEV:.6f}")
print(f"  gamma = {gamma_CEV:.6f}")

American put (CEV dynamics) at X0=100:
  price = 5.918298
  delta = -0.480640
  gamma = 0.026400


# 2d

Without changing the `price_put_CEV` code, pass in a `CEV` dynamics object that encodes Black-Scholes: set $\alpha = 0$ (so $X^{2(1+\alpha)} = X^2$), the volcoeff equal to BS $\sigma = 0.30$, and $\text{rGrow} = r = 0.05$ (so under risk-neutral measure, a stock has drift $rX$).

In [50]:
hw42_bs_dynamics = CEV(volcoeff=0.30, alpha=0, rGrow=0.05, r=0.05, X0=100)

(X0_bs, putprice_bs) = hw42FD.price_put_CEV(hw42contract, hw42_bs_dynamics)

i100_bs = np.argmin(np.abs(X0_bs - 100.0))
print(f"American put (Black-Scholes dynamics) at X0=100: {putprice_bs[i100_bs]:.6f}")

American put (Black-Scholes dynamics) at X0=100: 5.441980


# 2e

Under Black-Scholes, volatility is constant, so the European implied-vol curve is **flat** in strike.

Under CEV with $\alpha = -0.5$, the instantaneous volatility of $X$ is $\sigma_{\text{CEV}}(X) = \text{volcoeff} \cdot X^\alpha = 3/\sqrt{X}$, which **decreases** as $X$ increases. So states where the underlying has fallen are states of higher volatility, and states where the underlying has risen are states of lower volatility. Translating this into the European implied-vol surface:
- **Low strikes** (OTM puts / deep-ITM calls, where the relevant terminal $X$ is low) see higher vol → **higher** implied vol.
- **High strikes** see lower vol → **lower** implied vol.

So the CEV model with $\alpha = -0.5$ generates a **downward-sloping** volatility skew (implied vol falling with strike), qualitatively similar to the "equity skew" observed in real markets — in contrast to the flat BS skew.